# Advance RAG MultiDoc Chatbot - Complete GPU Benchmark & Test Suite

This notebook runs the complete evaluation harness and test suite on Google Colab with GPU acceleration (T4 / A100 / V100):
1. **Full Pytest Test Suite** (254 unit & integration tests)
2. **Corpus Ingestion** (24 travel policies & distractors into Qdrant & BM25)
3. **GPU-Accelerated RAG Benchmark** (162 cases across Dense, BM25, Hybrid RRF, and Cross-Encoder Reranker)
4. **RRF Hyperparameter Tuning** (Grid sweep on Dev split, validation on Heldout with 95% bootstrap CIs)
5. **Relevance Threshold Calibration** (Optimized on Dev subject to <=5% false abstention, evaluated on Heldout with Wilson 95% CIs)
6. **README Sync Verification** (Automated generation & CI anti-tamper check)

### Step 1: Check GPU Acceleration
Ensure you have selected **Runtime > Change runtime type > T4 GPU** (or A100).

In [ ]:
!nvidia-smi

### Step 2: Clone Repository & Install Dependencies

In [ ]:
# Clone the exact repository and latest commit
!git clone https://github.com/ak0000007/Advance_RAG_MultiDoc_Chatbot.git
%cd Advance_RAG_MultiDoc_Chatbot

# Install dependencies
!pip install -q -r requirements.txt
!pip install -q pytest pytest-asyncio

### Step 3: Configure Environment File (.env)
Configures embedded local Qdrant and BM25 store. External API keys are optional for retrieval benchmarking.

In [ ]:
env_content = """
QDRANT_URL=""
BM25_STORE_PATH="./data/bm25_index.pkl"
COLLECTION_NAME="multidoc_rag"
RERANKER_MODEL_NAME="BAAI/bge-reranker-v2-m3"
USE_RERANKER=true
LANGSMITH_TRACING=false
CORS_ORIGINS=["http://localhost:3000","http://localhost:8000","http://localhost:5173"]
"""
with open(".env", "w", encoding="utf-8") as f:
    f.write(env_content.strip())

print(".env successfully configured for GPU evaluation.")

### Step 4: Run Full Pytest Suite (247 Tests)
Runs all unit tests, dataset integrity checks, judge mocks, and sync verifications.

In [ ]:
!pytest -q

### Step 5: Ingest Expanded 24-Document Corpus
Embeds all 16 authoritative policies and 8 distractor documents into embedded Qdrant and synchronizes BM25 store.

In [ ]:
!python ingest.py --recreate

### Step 6: Run GPU-Accelerated Live RAG Benchmark
Runs all 4 pipelines on CUDA GPU. On T4 GPU, cross-encoder reranking takes ~15ms per query instead of ~21s on CPU.

In [ ]:
!python -m evals.rag.runner --repeats 3 --seed 0

### Step 7: Run RRF Grid Tuning (Dev Split -> Heldout Split)
Sweeps dense weights and RRF constant k on Dev split, evaluating the best combination on the Heldout split with 95% bootstrap CIs.

In [ ]:
!python scripts/tune_rrf.py

### Step 8: Calibrate Relevance Abstention Threshold
Optimizes semantic threshold on Dev split to maximize true abstention on unanswerables while bounding false abstention to <= 5%, reporting Wilson 95% CIs on Heldout.

In [ ]:
!python scripts/calibrate_threshold.py --signal dense_cosine --max-false-abstention 0.05

### Step 9: Render & Verify README Benchmark Tables

In [ ]:
!python scripts/render_readme_tables.py
!python scripts/check_readme_sync.py

### Step 10: Download Generated Evaluation Reports

In [ ]:
from google.colab import files
files.download("evals/results/rag_report.json")
files.download("evals/results/rrf_tuning_report.json")
files.download("evals/results/threshold_calibration_report.json")